# 08 · Nieuwe gegevens toevoegen en een voederstrategie beoordelen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De toestellen blijven intussen meten, dus je wilt het overzicht op de kantoorcomputer kunnen bijwerken met nieuwe meetdagen. In dit laatste notebook voeg je de resterende meetdagen toe aan je eigen brondatabase, stuur je ze via dezelfde MQTT- en API-route door en controleer je of alles netjes op zijn plaats komt.

Daarna vergelijk je de eerdere uitkomsten uit notebook 07 met de nieuwe periode en met de volledige proefduur. Je bekijkt per hok het gewichtsverloop, de groei, de voederopname en de voederconversie, onderzoekt welke voederstrategie bij elk hok zou kunnen passen en formuleert een onderbouwde keuze voor het bedrijf.

**Nodig:** een afgerond notebook 07, dezelfde Codespace en eigen branch, je werkdatabase uit notebook 02 en samenwerking met een andere gebruiker met de andere gegevensrol. Beide servers en hun ontvangers moeten bereikbaar zijn. De beginsituatie uit notebook 07 blijft bewaard als vergelijkingspunt.

Je levert dit ingevulde notebook in samen met de automatisch gemaakte tabellen, figuren, een controlebestand en een databasesnapshot. Je hoeft dus geen apart verslag te schrijven.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Tip bij heruitvoeren.** Gebruik een beginsituatie uit notebook 07 die met dezelfde rekenmethode (vijf individuele gewichtsmetingen per grens en `FILTER = '0'`) is gemaakt. Werk je met een oudere map `beginsituatie_07`, hernoem die dan eerst (bijvoorbeeld naar `beginsituatie_07_oud`), draai notebook 07 opnieuw en start daarna pas notebook 08.

</div>

## 1. De beginsituatie terugvinden

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

De bewaarde beginsituatie laat zien wat je al wist vóórdat de nieuwe meetdagen erbij kwamen. Een vergelijking klopt alleen als de gebruikte rekenregels en tijdsgrenzen precies bekend blijven. De volgende cel leest daarom eerst je beginsituatie uit notebook 07 in en controleert of alles overeenkomt met `07_analysecontrole.json`.

Gebruik dezelfde Python-kernel als in notebook 07. Ontbreekt `openpyxl` in een oudere Codespace, voer dan eerst de tussenstap uit om die Excel-lezer te installeren.

</div>

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl>=3.1,<4"])

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from contextlib import closing
from getpass import getpass
import hashlib, ipaddress, json, re, sqlite3, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

hoofdmap = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
inlever = hoofdmap / "gegevens/inlevering"
werk = hoofdmap / "gegevens/werk"
instellingen = json.loads((inlever / "01_werkplekcontrole.json").read_text(encoding="utf-8"))
deelnemer, rol = instellingen["deelnemerscode"], instellingen["rol"]
if branch != "student/" + deelnemer or instellingen.get("branch") != branch or rol not in ("gewichten", "voederopname"):
    raise ValueError("Controleer je eigen branch en rol uit notebook 01.")
sys.path[:0] = [str(hoofdmap / "programmatuur"), str(hoofdmap / "server")]
from aanvulling_08 import lees_excel, voeg_toe, lees_basis, vergelijk_basis, uitgebreide_analyse, snapshot
from analyse_07 import lees_analyse, vat_hokken_samen
from figuren_07 import gewichtsfiguur, intervalfiguur, hokvolgorde, procestijd, procesas
from publiceren import lees_metingen, meettopic, MqttSessie
from beginset_publiceren import publiceer_beginset
from meetgegevens import selectie_manifest
from api_ophalen import ApiClient, werk_analyse_bij

basisinfo, basistabellen = lees_basis(inlever / "beginsituatie_07")
controle07 = json.loads((inlever / "07_analysecontrole.json").read_text(encoding="utf-8"))
if controle07["deelnemerscode"] != deelnemer or controle07["basis"] != basisinfo:
    raise ValueError("Je controlebestand en beginsituatie uit 07 horen niet bij elkaar.")
bron = "asserva" if rol == "gewichten" else "schauer"
bronpad = werk / (bron + ".db")
analysepad = werk / "analyse_06.db"
maakcontrole = json.loads((hoofdmap / "gegevens/aanvulling/maakcontrole.json").read_text(encoding="utf-8"))
excelpad = hoofdmap / "gegevens/aanvulling" / (bron + "_aanvullende_metingen.xlsx")
server = json.loads((werk / "server.json").read_text(encoding="utf-8"))
if server["deelnemerscode"] != deelnemer:
    raise ValueError("Gebruik je eigen serverinstellingen.")
serveradres = str(ipaddress.IPv4Address(server["ipv4"]))
ca = werk / "ca.crt"
print("Rol:", rol, "| Laatste basisdag: 20 september 2023")
display(pd.DataFrame(basisinfo["bronnen"]).T[["aantal"]])
display(basistabellen["hokresultaten.csv"][["hok", "interval", "dieren_bruikbaar", "voederconversie"]].head())

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>De beginsituatie bevat 80.028 gewichtsrecords en 146.649 voederrecords over 101 gekoppelde dieren in tien hokken. Niet elk dier telt in elk interval mee. Die opgeslagen bestanden blijven gewoon staan als vast vergelijkingspunt, ook als je dit notebook later nog eens opnieuw uitvoert.</div>

## 2. De aanvulling in Excel onderzoeken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen laptop (Excel of een vergelijkbaar spreadsheetprogramma)

Download uit `gegevens/aanvulling/` het Excel-bestand dat bij jouw rol hoort en open het alleen om te bekijken. Het tabblad `toelichting` legt de kolommen uit. De tabbladen `week_5` (21–27 september) en `resterende_dagen` (28 september–5 oktober) samen bevatten vijftien extra kalenderdagen.

**Stap 1.** Bekijk de eerste rij van beide tabbladen en zoek `ROW_ID`, `EID`, `PEN`, `DATE` en jouw meetkolom (`VBW` of `VFI`). Klik op een cel in `ROW_ID` en bekijk de waarde in de formulebalk: die lange code is als tekst opgeslagen zodat geen cijfers worden afgerond.

**Stap 2.** Vergelijk enkele waarden met de uitleg op `toelichting` en zoek ook `ERROR` en `FILTER` op. De aanvulling bevat dus weer alle oorspronkelijke kwaliteitsvlaggen, want inlezen en filteren voor de analyse blijven twee aparte stappen.

**Stap 3.** Sluit Excel **zonder op te slaan**, want in Codespaces lees je rechtstreeks het oorspronkelijke bestand uit de repository. De volgende cel leest beide tabbladen in, controleert de bestandshash en toont de kolommen die je nodig hebt.

</div>

In [ ]:
rijen, bladoverzicht = lees_excel(excelpad, maakcontrole[bron])
display(bladoverzicht)
meetveld = "VBW" if rol == "gewichten" else "VFI"
voorbeeld = pd.DataFrame(rijen[:5])
display(voorbeeld[["ROW_ID", "EID", "PEN", "DATE", meetveld, "FILTER"]])
print("Aantal aanvullende records:", len(rijen))

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Wat kan door de aanvulling veranderen aan je overzicht, en wat verwacht je bij een herberekening van precies dezelfde oude periode?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

*Voeg hier je antwoord in. Tip: Maak onderscheid tussen meer records, een langere periode en gewijzigde oude waarden.*

</div>

## 3. Toevoegen aan de computer van het toestel

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

Jouw werkdatabase staat symbool voor de computer van het weeg- of voedertoestel. Daar voeg je nu eerst de nieuwe meetdagen aan toe, zodat ze straks via MQTT naar de server kunnen worden gestuurd en via de API op de kantoorcomputer belanden.

De cel hieronder maakt eerst een back-up van je werkdatabase en voegt daarna de nieuwe Excel-rijen in één transactie toe. Als een bestaand record met een andere inhoud zou botsen, wordt de hele toevoeging teruggedraaid. Records die al bestonden met dezelfde inhoud worden herkend en blijven één keer staan.

Controleer of het getoonde doelbestand jouw eigen `gegevens/werk/asserva.db` of `gegevens/werk/schauer.db` is. De tabel `hokken` blijft gewoon behouden. Vervang dit bijgewerkte bestand later dus niet door een oudere kopie van je laptop.

</div>

In [ ]:
print("Toevoegen aan:", bronpad.relative_to(hoofdmap))
backuppad = werk / (bron + "_voor_08.db.gz")
if not backuppad.exists():
    snapshot(bronpad, backuppad)
importcontrole = voeg_toe(bronpad, rijen, maakcontrole[bron])
display(pd.DataFrame([importcontrole]))
herhaalimport = voeg_toe(bronpad, rijen, maakcontrole[bron])
assert herhaalimport["nieuw"] == 0
assert herhaalimport["na"] == importcontrole["na"]
with closing(sqlite3.connect(bronpad.as_uri() + "?mode=ro", uri=True)) as db:
    display(pd.read_sql_query("SELECT MIN(DATE) AS eerste_datum, MAX(DATE) AS laatste_datum, COUNT(*) AS records FROM metingen", db))
print("Herhaalde import:", herhaalimport)

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>De eerste keer komen er 52.194 gewichtsrecords of 86.952 voederrecords bij, zodat het totaal op 132.222 gewichtsrecords of 233.601 voederrecords uitkomt. Voer je de cel later nog eens uit, dan moet het aantal nieuwe records nul zijn als teken dat alle eerdere toevoegingen behouden zijn.</div>

## 4. Beide gegevensstromen opnieuw laten aankomen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** samen met de andere gebruiker, in Codespaces en op je eigen DigitalOcean-server

Controleer eerst of beide ontvangers op de servers actief zijn, anders kan een broker nieuwe berichten missen voordat de ontvanger luistert. Je gebruikt dezelfde accounts en certificaten als in notebooks 04–06 en maakt niets opnieuw aan.

**Stap 1.** Open in Codespaces een terminal en maak verbinding via de geprinte SSH-opdracht hieronder, met je **`SSH beheer-wachtwoord`**.

</div>

In [ ]:
print(f"ssh -o StrictHostKeyChecking=ask beheer@{serveradres}")

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je eigen DigitalOcean-server (SSH-terminal)

**Stap 2.** Start (of controleer) alle vier de diensten handmatig op je server. Met `start` gebeurt er niets verkeerds als een dienst al draait, en na een eventuele herstart van je server zet je zo ook Mosquitto weer aan. De API hoort nog altijd bij de dienstnaam `iot-status`.

</div>

```bash
sudo systemctl start mosquitto iot-ontvanger iot-status nginx
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Als `sudo` om een wachtwoord vraagt, gebruik je **`SSH beheer-wachtwoord`**. Wacht tot de opdracht klaar is. Controleer daarna de toestand met de volgende afzonderlijke opdracht.

</div>

```bash
sudo systemctl is-active mosquitto iot-ontvanger iot-status nginx
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht vier keer `active`. De andere gebruiker doet precies hetzelfde op zijn of haar server. Krijg je ergens geen `active`, los dat dan eerst op met de hersteltips uit notebooks 05 en 06 voordat je berichten verstuurt. Sluit daarna de SSH-sessie netjes af.

</div>

```bash
exit
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

**Stap 3.** Spreek samen met de andere gebruiker af dat beide ontvangers klaarstaan en voer dan pas de volgende cel uit. Deze cel stuurt alleen de nieuwe records vanaf 21 september door, zodat je de eerste vier weken niet opnieuw hoeft te versturen.

Bij het starten verschijnt **bovenaan het scherm** het verborgen invoerveld voor **`MQTT-broker eigenaar`**. De publicatie werkt in groepen van honderd berichten en toont de voortgang. Een brokerbevestiging betekent nog alleen dat de broker het bericht aangenomen heeft: of alles ook in de database op de server staat, controleer je in stap 5. Wordt het versturen onderbroken, dan mag je dezelfde cel gewoon opnieuw uitvoeren.

</div>

In [ ]:
def aanvullende_berichten():
    return (b for b in lees_metingen(bronpad, rol, deelnemer, None)
            if "2023-09-21" <= b["bronrecord"]["DATE"] < "2023-10-06")

verwacht_eigen = selectie_manifest(lees_metingen(bronpad, rol, deelnemer, None))
wachtwoord = getpass("MQTT-broker eigenaar: ")
try:
    with MqttSessie(serveradres, ca, wachtwoord) as sessie:
        gepubliceerd = publiceer_beginset(sessie, meettopic(deelnemer, rol), aanvullende_berichten())
finally:
    del wachtwoord
assert gepubliceerd == maakcontrole[bron]["aantal"]
broncontrole = {"deelnemerscode": deelnemer, "rol": rol, "bronnen": verwacht_eigen}
(inlever / "08_eigen_broncontrole.json").write_text(json.dumps(broncontrole, indent=2) + "\n", encoding="utf-8")
print("Bewaar en deel 08_eigen_broncontrole.json met je partner.")

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 4.** Download `gegevens/inlevering/08_eigen_broncontrole.json` en bezorg dat bestand aan de andere gebruiker (het bevat alleen aantallen en inhoudsvingerafdrukken, geen wachtwoorden). Upload het bestand dat je van de andere gebruiker ontvangt naar `gegevens/werk/` onder de exacte naam **`08_partner_broncontrole.json`**.

**Stap 5.** Voer de controlecel hieronder pas uit zodra beide publicaties klaar zijn en geef bovenaan het scherm je **`API-Droplet wachtwoord`** in. De API leest dan de verzameldatabase op je server en vergelijkt de aantallen en vingerafdrukken met beide broncontroles. Is de ontvangst bij een van beiden nog bezig, wacht dan heel even en voer de cel opnieuw uit.

</div>

In [ ]:
partner = json.loads((werk / "08_partner_broncontrole.json").read_text(encoding="utf-8"))
andere_bron = "schauer" if bron == "asserva" else "asserva"
andere_rol = "voederopname" if rol == "gewichten" else "gewichten"
if partner["deelnemerscode"] == deelnemer or partner["rol"] != andere_rol or set(partner["bronnen"]) != {andere_bron}:
    raise ValueError("Gebruik de actuele broncontrole van je partner met de andere rol.")
verwacht = {**verwacht_eigen, **partner["bronnen"]}
assert verwacht["asserva"]["aantal"] == 132222 and verwacht["schauer"]["aantal"] == 233601
api = ApiClient("https://" + serveradres, ca, getpass("API-Droplet wachtwoord: "))
try:
    servermeta = api.ophalen("/metadata")
    if servermeta["bronnen"] != verwacht:
        raise ValueError("De serverinhoud is nog niet gelijk aan beide bijgewerkte bronnen. Controleer de ontvangers en publiceer zo nodig opnieuw.")
finally:
    api.sluiten()
display(pd.DataFrame(servermeta["bronnen"]).T[["aantal"]])

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Waarom zijn zowel de brokerbevestiging als de servercontrole nodig?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

*Voeg hier je antwoord in. Tip: Benoem welk programma de SQL-records opslaat.*

</div>

## 5. De kantoorcomputer bijwerken via de API

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

Je haalt de verzamelde metingen nu binnen met dezelfde lees-API als in notebook 06. De import doorloopt de volledige API om ook de eerdere inhoud te verifiëren, en voegt alleen nieuwe records toe aan `gegevens/werk/analyse_06.db`. Die bestandsnaam blijft gelijk omdat het dezelfde analysedatabase is (de controlebestanden uit 06 en 07 bewaren immers de vroegere tussenstand).

Geef bovenaan het scherm opnieuw je **`API-Droplet wachtwoord`** in en laat de cel rustig uitlopen. Spreek af dat de gegevens op de server tijdens het ophalen niet gewijzigd worden. Loopt er iets fout, dan draait de import die ronde automatisch terug.

</div>

In [ ]:
api = ApiClient("https://" + serveradres, ca, getpass("API-Droplet wachtwoord: "))
try:
    api_import = werk_analyse_bij(api.ophalen, analysepad)
    assert api_import["bronnen"] == verwacht
    api_herhaling = werk_analyse_bij(api.ophalen, analysepad)
    assert api_herhaling["nieuw"] == 0 and api_herhaling["bronnen"] == verwacht
finally:
    api.sluiten()
data, eindmanifest = lees_analyse(analysepad)
print("Totaal op de kantoorcomputer:", len(data))
print("Nieuwe records bij deze API-import:", api_import["nieuw"])
print("Nieuwe records bij herhaling:", api_herhaling["nieuw"])

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Controle</strong><br>Bij de eerste volledige aanvulling komen er 139.146 nieuwe records bij, waarmee `analyse_06.db` op 365.823 records komt. Voer je de cel nog een keer uit, dan moeten er nul nieuwe records bijkomen en blijven de totalen en vingerafdrukken gelijk. (De Excel-import zelf gebeurde alleen in je brondatabase.)</div>

## 6. Controleren wat gelijk blijft en wat verandert

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

Eerst reken je de vier oorspronkelijke intervallen uit notebook 07 opnieuw uit over precies dezelfde begindata, en vergelijk je de populatie, daggewichten en dierintervallen met je opgeslagen beginsituatie. Omdat de CSV-bestanden twaalf significante cijfers bewaren, gebruiken we bij het vergelijken van kommagetallen een kleine tolerantie van 10⁻¹⁰.

Daarna breid je de berekening uit naar de volledige periode. De dagmedianen dienen nog altijd alleen voor de gewichtslijn, terwijl elk grensgewicht het gemiddelde is van de vijf dichtstbijzijnde individuele wegingen en de opname de som is van alle geldige bezoeken in dat interval. Door de langere periode kan het aantal bruikbare dieren per hok wel verschuiven, dus houden we die tellingen zichtbaar naast elk resultaat.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Dezelfde procestijd als in notebook 07.** Het allereerste meetmoment uit de oorspronkelijke twee bronnen blijft dag 0, ook nu er latere weken zijn toegevoegd. De stippellijnen markeren opnieuw dag 10 en dag 18. Alle kalendergrenzen van de oude intervallen blijven behouden zodat de vergelijking met notebook 07 zuiver blijft. Alleen in de aparte voederfasen-vergelijking verderop gebruiken we grenzen op hele procesdagen. Elk punt op een intervalgrafiek staat in het midden van zijn meetperiode, met een horizontaal lijntje voor de duur.

</div>

In [ ]:
from figuren_07 import procestijd, procesas
processtart = pd.to_datetime(data["tijd"]).min()
print("Procestijd 0:", processtart)
voorbeeld_tijd = pd.to_datetime(data["tijd"]).head(5)
display(pd.DataFrame({"tijdstip": voorbeeld_tijd,
                      "procestijd_dagen": procestijd(voorbeeld_tijd, processtart)}))
for dagnummer in (10, 18):
    print(f"Dag {dagnummer}:", processtart + pd.Timedelta(days=dagnummer))

In [ ]:
display(data.groupby(["bron", "vlag"], dropna=False).size().rename("rijen").reset_index())
print("Alle berekeningen hieronder gebruiken uitsluitend FILTER = '0'.")
oude_hokken = vergelijk_basis(data, basisinfo, basistabellen)
print("Beginsituatie opnieuw berekend: gelijk gebleven.")
resultaat = uitgebreide_analyse(data, basistabellen["populatie.csv"])
display(resultaat["audit"])
display(resultaat["dierintervallen"].groupby(["interval", "vanaf", "tot"]).agg(
    dieren=("dier_id", "size"), bruikbaar=("bruikbaar", "sum")))

| Vergelijking | Begin inclusief | Einde exclusief | Betekenis |
|---|---|---|---|
| Basis | 25 augustus | 20 september | Eén berekening over de oude analyseperiode |
| Aanvulling | 21 september | 5 oktober | Eén berekening over de nieuwe volledige dagen |
| Geheel | 25 augustus | 5 oktober | Eén berekening over de langere periode |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De allereerste en allerlaatste meetdag van de proef zijn onvolledig. De analyse start daarom op 25 augustus, en op 5 oktober helpen de wegingen nog om het eindgewicht om 00:00 te schatten, zonder die onvolledige voederdag zelf mee te tellen.

Omdat de oude eindgrens op 20 september lag en de aanvulling op 21 september begint, verschijnt er in de tijdgrafiek één kort overgangsinterval van 20 tot 21 september. Zo blijft elke oude grens herkenbaar zonder dat er een dag wordt overgeslagen. Besef wel dat een interval van één dag veel gevoeliger is voor schommelingen dan periodes van vijf of zeven dagen.

De cijfers voor de volledige proefperiode worden rechtstreeks uit de totale opname en totale groei berekend, dus niet door losse intervalconversies te middelen. Ontbrekende voederdagen worden getoond zonder een dier meteen uit te sluiten, maar bij een volledig lege periode of niet-positieve groei is er geen bruikbare voederconversie.

</div>

In [ ]:
perioden = resultaat["hokperioden"]
kolommen = ["hok", "periode", "dagen", "dieren_bruikbaar", "dieren_populatie", "groei_g_dier_dag", "voeder_g_dier_dag", "voederconversie"]
display(perioden[kolommen].round(3))
verschil = perioden.pivot(index="hok", columns="periode", values="voederconversie")
verschil["verschil_geheel_min_basis"] = verschil["geheel"] - verschil["basis"]
display(verschil.round(3))

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Ontbrekende uitkomst</strong><br>Doordat we de vijf dichtstbijzijnde gewichten gebruiken, krijgt een dier ook een schatting als de metingen maar aan één kant van de grens liggen. Controleer dus altijd de tijdafstanden in de tabellen. Ontbreekt een uitkomst toch, dan komt dat door minder dan vijf wegingen, geen enkel geldig voederbezoek of een groei kleiner dan of gelijk aan nul. De tabel hieronder toont de exacte redenen.</div>

In [ ]:
display(resultaat["dierperioden"].loc[~resultaat["dierperioden"].bruikbaar]
        .groupby(["periode", "hok", "reden"]).size().rename("dieren").reset_index())

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Waarom kan de conversie over de gehele periode veranderen terwijl alle vier oude intervalresultaten gelijk blijven?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

*Voeg hier je antwoord in. Tip: Gebruik de periodegrenzen en de aantallen bruikbare dieren.*

</div>

## 7. Een overzicht voor alle hokken maken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

Je maakt nu drie samenhangende figuren over de uitgebreide periode: het gewichtsverloop, de groei en voederopname in kilogram per bruikbaar dier per dag, en de voederconversie per interval.

Alle hokken gebruiken binnen een figuur dezelfde schaal, zodat je ze eerlijk naast elkaar kunt leggen. Kijk daarbij altijd ook naar het aantal bruikbare dieren bij elk punt, want een mooi cijfer op basis van weinig dieren zegt veel minder over het hele hok. De gewichtsfiguur gebruikt alle gewogen dieren per dag en kan dus iets anders samengesteld zijn dan de intervalcijfers.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Wat betekent bijvoorbeeld `8/10` dieren?** De teller geeft aan voor hoeveel dieren de conversie berekend kon worden, en de noemer hoeveel dieren oorspronkelijk aan dat hok gekoppeld zijn. Groei, voederopname en conversie gebruiken precies dezelfde groep bruikbare dieren. Dit is geen telling van levende of gestorven biggen.

Per periodegrens gebruiken we de vijf dichtstbijzijnde individuele wegingen, zonder maximumgrens van 48 uur. Alle geldige voederbezoeken in het interval worden opgeteld, ook als er een dag tussenuit valt. Een dier valt dus alleen af bij te weinig gewichtsmetingen, nul voederbezoeken in de hele periode of niet-positieve groei. Bij grote tijdafstanden moet je zelf beoordelen of een geschat gewicht nog representatief is.

</div>

In [ ]:
figuren = {
    "08_gewichtsverloop.png": gewichtsfiguur(resultaat["dag"], start=processtart),
    "08_groei_en_voederopname.png": intervalfiguur(resultaat["hokintervallen"], kilogram=True, start=processtart),
    "08_voederconversie.png": intervalfiguur(resultaat["hokintervallen"], "conversie", start=processtart),
}
for naam, fig in figuren.items():
    fig.savefig(inlever / naam, dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Doe-opdracht.** Kies twee hokken die je wilt vergelijken en vul hun hoknummers in bij `mijn_hokken`, samen met een periode bij `mijn_periode` (`basis`, `aanvulling` of `geheel`). Neem bij voorkeur minstens één hok dat je in notebook 07 al bekeek, en vergelijk dekking, groei, voederopname en conversie samen.

</div>

In [ ]:
# Kies twee verschillende hoknummers als tekst, bijvoorbeeld ["1", "9"].
# Kies daarna een van de drie perioden.
# mijn_hokken = ...
# mijn_periode = ...

In [ ]:
assert len(set(mijn_hokken)) == 2 and set(mijn_hokken) <= set(perioden.hok)
assert mijn_periode in {"basis", "aanvulling", "geheel"}
eigen_selectie = perioden[perioden.hok.isin(mijn_hokken) & perioden.periode.eq(mijn_periode)]
display(eigen_selectie[kolommen].round(3))
for r in eigen_selectie.itertuples():
    print(f"Hok {r.hok}: {r.groei_g_dier_dag:.1f} g groei/dier/dag, "
          f"{r.voeder_g_dier_dag:.1f} g voeder/dier/dag, conversie {r.voederconversie:.3f}, "
          f"{r.dieren_bruikbaar}/{r.dieren_populatie} dieren.")

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Welk verschil tussen de twee gekozen hokken vind je relevant voor de bedrijfsvoering?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

*Voeg hier je antwoord in. Tip: Noem concrete waarden uit de tabel, de periode, dekking en een mogelijke andere verklaring.*

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Facultatieve verkenning</strong><br>Herhaal deze vergelijking eens met alleen de dieren die zowel in de basisperiode als in de aanvulling bruikbaar zijn. Zo zie je of een verschil komt door een andere diersamenstelling. Je hoofdresultaten blijven onaangetast.</div>

In [ ]:
# Facultatief: gebruik resultaat["dierperioden"].
# Zoek de dier_id's met bruikbaar == True in zowel basis als aanvulling.
# Selecteer die dieren en gebruik vat_hokken_samen per periode.
# Vergelijk vooral dieren_bruikbaar en voederconversie met de hoofdroute.

## 8. Van hokpatronen naar een voederkeuze

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je combineert nu de gegevens van twee aparte meetsystemen om een praktische bedrijfskeuze te onderbouwen die je met alleen weeg- of alleen voedergegevens niet goed kunt maken. De vier mogelijke strategieën verschillen in het eiwitgehalte van het startvoer en in het moment waarop naar het vervolgvoer wordt overgeschakeld.

</div>

| Keuze | Eerste voeder (g ruw eiwit/kg) | Overgang naar vervolgvoeder | Vervolgvoeder (g/kg) |
|---|---:|---|---:|
| HRE10 | 188 | Na 10 dagen | 179 |
| HRE18 | 188 | Na 18 dagen | 179 |
| LRE10 | 162 | Na 10 dagen | 179 |
| LRE18 | 162 | Na 18 dagen | 179 |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een verschil in groei kan wijzen op een van de voederstrategieën, al zie je het eiwitgehalte natuurlijk niet rechtstreeks in de cijfers terug. De vroege intervallen rond dag 10 en dag 18 zijn het informatiefst voor het omslagmoment, want later krijgen alle hokken hetzelfde vervolgvoer. Een achterstand kan dan bijvoorbeeld worden ingehaald, zonder dat dat bewijst welk voer een hok kreeg.

**Doe-opdracht.** Koppel aan elk van de tien hokken de best passende vermoedelijke strategie en noteer je argumenten en eventuele twijfels. We weten dat er in totaal drie `HRE18`-hokken, drie `LRE18`-hokken, twee `HRE10`-hokken en twee `LRE10`-hokken zijn: controleer of jouw indeling klopt met die aantallen. Beschrijf per hok welk patroon je ziet en geef je zekerheid aan met `laag`, `matig` of `hoog`. Baseer je op je metingen en grafieken, niet op een vooraf bekende sleutel.

Kies daarna één van de vier strategieën als bedrijfsadvies voor een volgende ronde. Geef aan of je die meteen breed zou invoeren of eerst beperkt wilt testen, wat je op basis van de cijfers verwacht, en welke ontbrekende gegevens (zoals voerprijzen, diergezondheid, uitval en afleverdoelen) je eerst nog zou willen weten.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**De voerwissels als periodegrenzen.** `HRE` staat voor hoog ruw eiwit en `LRE` voor laag ruw eiwit. De getallen `10` en `18` geven de dag van de overgang aan vanaf dag 0 (24 augustus 2023 om 12:00): dag 10 valt op 3 september om 12:00 en dag 18 op 11 september om 12:00. Dat zijn de analysegrenzen ten opzichte van het eerste meetpunt, geen exacte uurregistraties van de voerbak.

Tussen dag 10 en dag 18 krijgen de vroeg gewisselde groepen al het gewone vervolgvoer, terwijl de laat gewisselde groepen nog op hun startvoer zitten. Juist die fase is dus nuttig om een verschil tussen `HRE18` en `LRE18` te verkennen. We gebruiken dezelfde rekenregels als in notebook 07, slaan de eerste onvolledige 24 uur over en sluiten af op dag 27 (nog binnen de eerste vier weken).

**Stap 1.** Vergelijk dag 1–10, dag 10–18 en dag 18–27 en let telkens op groei, opname, conversie en het aantal bruikbare dieren. Groei en voederopname staan hier in kilogram per dier per dag.

</div>

In [ ]:
from analyse_07 import selecteer, daggewichten, bereken_intervallen
gewichten_fase, voer_fase, populatie_fase, _, _ = selecteer(data)
dag_fase = daggewichten(gewichten_fase)
proefstart = processtart
grenzen = [proefstart + pd.Timedelta(days=d) for d in (1, 10, 18, 27)]
fase_dieren = bereken_intervallen(gewichten_fase, voer_fase, populatie_fase, grenzen)
fasen = vat_hokken_samen(fase_dieren)
fasen["groei_kg_dier_dag"] = fasen.groei_g_dier_dag / 1000
fasen["voeder_kg_dier_dag"] = fasen.voeder_g_dier_dag / 1000
fasekolommen = ["hok", "interval", "vanaf", "tot", "dieren_bruikbaar", "dieren_populatie",
                "groei_kg_dier_dag", "voeder_kg_dier_dag", "voederconversie"]
display(fasen[fasekolommen].round(3))
for soort in ("groei_voeder", "conversie"):
    fig = intervalfiguur(fasen, soort, kilogram=True, start=processtart)
    fig.savefig(inlever / ("08_voederfasen_" + soort + ".png"), dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)
rangorde = fasen[fasen.interval.eq(2)].sort_values("groei_kg_dier_dag", ascending=False)
display(rangorde[fasekolommen].round(3))

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Aantal dieren en kwaliteit van de grensgewichten.** De eerste tabel toont het aantal bruikbare dieren per hok en fase. De tabellen daaronder laten eventuele uitsluitingsredenen en de tijdafstanden van de vijf wegingen zien. Ontbrekende voederdagen worden wel vermeld, maar sluiten een dier niet uit.

Let ook bij dieren die wél meetellen op hoe ver de gebruikte gewichten van de grens af liggen. Als een dier halverwege niet meer op de weegschaal kwam, kunnen dezelfde oude wegingen voor meerdere grenzen gekozen worden, wat bijna nul groei oplevert. De code detecteert zelf geen uitval of sterfte.

</div>

In [ ]:
display(fasen[["hok", "interval", "dieren_bruikbaar", "dieren_populatie"]])
uitsluitingen = fase_dieren.loc[~fase_dieren.bruikbaar].copy()
redenen = uitsluitingen.assign(reden=uitsluitingen.reden.str.split(";")).explode("reden")
redennamen = {
    "begin_minder_dan_vijf_gewichten": "Minder dan vijf behouden gewichtsmetingen",
    "eind_minder_dan_vijf_gewichten": "Minder dan vijf behouden gewichtsmetingen",
    "geen_voederrecords": "Geen behouden voederregistratie in het interval",
    "geen_positieve_groei": "Geschatte groei niet positief",
}
redenen["uitleg"] = redenen.reden.map(redennamen).fillna(redenen.reden)
display(redenen.groupby(["interval", "uitleg"]).size().rename("dieren").reset_index())
display(uitsluitingen[["interval", "hok", "dier_id", "voederdagen", "dagen",
                       "afstand_begin_uren", "afstand_eind_uren", "reden"]])
# Toon ook de tijdafstand bij dieren die wel meetellen.
display(fase_dieren[["interval", "hok", "dier_id", "aantal_begin_metingen", "aantal_eind_metingen",
                      "afstand_begin_uren", "afstand_eind_uren", "voederdagen", "dagen", "bruikbaar"]])

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Groei en conversie samen bekijken.** De vier spreidingsgrafieken hieronder tonen één punt per hok, met de groei horizontaal en de voederconversie verticaal. Een hok dat rechtsonder staat, groeit dus relatief snel met relatief weinig voer per kilogram groei. Elk hok houdt in alle vier panelen dezelfde kleur. De aantallen dieren staan apart in een tabel zodat de punten goed leesbaar blijven.

De eerste drie panelen tonen de losse voederfasen, en het vierde paneel vat de hele periode van dag 1 tot 27 samen (dus niet de verlenging tot oktober). Groei en conversie worden voor dat hele blok opnieuw berekend uit de totalen en niet gemiddeld over de drie fasen, waardoor ook het aantal bruikbare dieren iets kan verschillen. Hokken zonder geldige uitkomst staan onderaan het paneel vermeld.

**Doe-opdracht.** Zoek twee hokken die in één fase dicht bij elkaar liggen, maar in een andere fase juist uit elkaar lopen. Gebruik hun groei, conversie en dieraantallen in je uitleg en geef aan of dit jouw hypothese steunt of juist twijfelachtig maakt. Punten die toevallig bij elkaar liggen, bewijzen nog niet dat ze hetzelfde voer kregen.

</div>

In [ ]:
totaal_dieren = bereken_intervallen(
    gewichten_fase, voer_fase, populatie_fase, [grenzen[0], grenzen[-1]])
totaal_hokken = vat_hokken_samen(totaal_dieren)
panelen_spreiding = [
    ("Dag 1–10", fasen[fasen.interval.eq(1)]),
    ("Dag 10–18", fasen[fasen.interval.eq(2)]),
    ("Dag 18–27", fasen[fasen.interval.eq(3)]),
    ("Geheel: dag 1–27", totaal_hokken),
]
fig_spreiding, assen = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True, layout="constrained")
fig_spreiding.suptitle("Groei en voederconversie per hok", fontsize=16)
clustergegevens = []
kleuren = {str(i): plt.get_cmap("tab10")(i-1) for i in range(1, 11)}
for ax, (titel, tabel) in zip(assen.flat, panelen_spreiding):
    tabel = tabel.copy()
    tabel["groei_kg_dier_dag"] = tabel.groei_g_dier_dag / 1000
    clustergegevens.append(tabel.assign(periode=titel))
    geldig = tabel.groei_kg_dier_dag.notna() & tabel.voederconversie.notna()
    punten = tabel[geldig]
    for r in punten.itertuples():
        ax.scatter(r.groei_kg_dier_dag, r.voederconversie,
                   color=kleuren[r.hok], s=60, edgecolor="white", linewidth=.5)
    ontbrekend = ", ".join(hokvolgorde(tabel.loc[~geldig, "hok"]))
    if ontbrekend:
        ax.text(.02, .02, "Niet berekenbaar: hok " + ontbrekend, transform=ax.transAxes, fontsize=9)
    ax.set(title=titel, xlabel="Gemiddelde groei (kg per dier per dag)",
           ylabel="Voederconversie (kg voeder per kg groei)")
    ax.grid(alpha=.2)
    ax.margins(x=.18, y=.18)
clustergegevens = pd.concat(clustergegevens, ignore_index=True)
from matplotlib.lines import Line2D
legenda = [Line2D([], [], marker="o", linestyle="", color=kleuren[str(i)], label=f"Hok {i}")
           for i in range(1, 11)]
fig_spreiding.legend(handles=legenda, loc="outside lower center", ncol=5, frameon=False)
display(clustergegevens[["periode", "hok", "dieren_bruikbaar", "dieren_populatie"]])
fig_spreiding.savefig(inlever / "08_groei_tegen_conversie.png", dpi=150, bbox_inches="tight")
display(fig_spreiding)
plt.close(fig_spreiding)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Dezelfde vier periodes als tabel naast elkaar.** Om de hokken op het oog makkelijk te rangschikken, toont de volgende cel dezelfde vier periodes als de spreidingsfiguur hierboven in twee overzichtstabellen: eerst de gemiddelde groei (in kg per dier per dag) en daarna de voederconversie (in kg voeder per kg groei), telkens met de tien hokken als rijen en de vier kolommen `Dag 1–10`, `Dag 10–18`, `Dag 18–27` en `Geheel: dag 1–27`.

Standaard staan de hokken in beide tabellen gerangschikt van hoogste naar laagste groei in `Dag 1–10`. Je kunt de rangschikking eenvoudig wisselen door bovenaan in de codecel deze drie instellingen aan te passen:
- `sorteer_op_tabel`: kies `"groei"` of `"voederconversie"`.
- `sorteer_op_kolom`: kies `"Dag 1–10"`, `"Dag 10–18"`, `"Dag 18–27"` of `"Geheel: dag 1–27"`.
- `oplopend`: zet op `False` voor hoogste eerst (handig voor groei) of `True` voor laagste eerst (handig voor voederconversie).

</div>

In [ ]:
periode_kolommen = ["Dag 1–10", "Dag 10–18", "Dag 18–27", "Geheel: dag 1–27"]

groei_tabel = clustergegevens.pivot(
    index="hok", columns="periode", values="groei_kg_dier_dag"
)[periode_kolommen]
conversie_tabel = clustergegevens.pivot(
    index="hok", columns="periode", values="voederconversie"
)[periode_kolommen]

# Pas deze drie instellingen aan om beide tabellen anders te rangschikken:
sorteer_op_tabel = "groei"       # "groei" of "voederconversie"
sorteer_op_kolom = "Dag 1–10"    # "Dag 1–10", "Dag 10–18", "Dag 18–27" of "Geheel: dag 1–27"
oplopend = False                 # False = hoogste eerst, True = laagste eerst

tabellen = {"groei": groei_tabel, "voederconversie": conversie_tabel}
if sorteer_op_tabel not in tabellen:
    raise ValueError('Kies "groei" of "voederconversie" voor sorteer_op_tabel.')
if sorteer_op_kolom not in periode_kolommen:
    raise ValueError(f"Kies een geldige periodekolom uit {periode_kolommen}.")

volgorde = tabellen[sorteer_op_tabel].sort_values(
    sorteer_op_kolom, ascending=oplopend
).index

print(f"Rangschikking: {sorteer_op_tabel} op '{sorteer_op_kolom}' ({'oplopend' if oplopend else 'aflopend'})")
print("\nTabel 1. Gemiddelde groei per hok (kg per dier per dag)")
display(groei_tabel.loc[volgorde].round(3))

print("Tabel 2. Voederconversie per hok (kg voeder per kg groei)")
display(conversie_tabel.loc[volgorde].round(3))

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Stap 2.** Gebruik eerst de groeisnelheid tussen dag 10 en dag 18 als voorlopige aanwijzing: de drie snelst groeiende hokken zijn kandidaten voor `HRE18` en de drie traagst groeiende voor `LRE18`. Kijk daarna of voederopname en conversie dat beeld bevestigen, want traag groeien omdat een hok weinig eet is iets heel anders dan traag groeien bij veel opname. Zo'n rangorde is een hulpmiddel om te verkennen, geen waterdicht recept.

**Stap 3.** Vergelijk voor de vier overblijvende hokken de eerste periode (dag 1–10) en kijk hoe de hokken na dag 18 verder evolueren. Wijs twee hokken voorlopig toe aan `HRE10` en twee aan `LRE10`. Vergeet niet dat leeftijd, gezondheid, startgewicht en welke dieren meetellen ook invloed hebben op de grafieken. In `fase_dieren` kun je nagaan of steeds dezelfde dieren meededen.

**Eigen onderzoek.** Kies één twijfelgeval en leg uit welke andere indeling je ook zou kunnen verdedigen. Je mag bijvoorbeeld zelf kortere periodes rond dag 10 of 18 testen of steeds precies dezelfde dieren volgen, zolang je maar op de hoofdresultaten voortbouwt en niet blijft schuiven met grenzen tot je een gewenste uitkomst ziet.

</div>

In [ ]:
hypothesen = pd.DataFrame({
    "hok": hokvolgorde(perioden.hok),
    "strategie": "",
    "zekerheid": "laag",
    "argument": "",
})
display(hypothesen)
# Vul argumenten en strategieën in met bijvoorbeeld:
# hypothesen.loc[hypothesen.hok == "1", "argument"] = "Mijn waarneming ..."
# Kies één strategie en formuleer je onderbouwing.
# gekozen_strategie = ...
# bedrijfsadvies = ...

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Facultatieve verdieping: lysine.** Op slide 29 van de lespresentatie staat een rekenvoorbeeld om de lysinebehoefte voor onderhoud en groei te vergelijken met de opname. Daarmee kun je verkennen of een lagere groei past bij het aanbod van aminozuren. Dit onderdeel is niet verplicht. Benoem bij zo'n berekening altijd je aannames, eenheden en onzekerheid, en presenteer geen schijnnauwkeurige benutting zolang de voersamenstelling per hok nog een hypothese is.

</div>

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Interpretatie</strong><br>Elke toewijzing blijft een onderbouwde hypothese: kies voor ieder hok je beste inschatting, maar benoem twijfel eerlijk. Dat de aantallen per strategie kloppen (3, 3, 2 en 2) bewijst nog niet dat elk los hok juist ingedeeld is. Zonder zekere behandelingslabels en extra bedrijfsdata blijft ook het economisch optimum onzeker.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Wat heeft de integratie van beide systemen aan je beslissing toegevoegd, en welke vraag blijft onbeantwoord?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

*Voeg hier je antwoord in. Tip: Verbind een berekende verhouding met een concreet ontbrekend gegeven.*

</div>

## 9. Resultaten automatisch bewaren en inleveren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 08)

De laatste cel bewaart al je berekende tabellen, hokhypothesen, bedrijfsadvies en technische controles, en maakt daarnaast via SQLite een gecomprimeerde back-up (`08_analysedatabase.db.gz`) zodat de volledige database mee kan in je inlevering. De beginsituatie uit notebook 07 blijft gewoon onaangeroerd.

Het automatische controlebestand legt vast welke gegevens en berekeningen gedraaid hebben, maar kan je inhoudelijke argumentatie niet zelf beoordelen. Jouw antwoorden en bedrijfsadvies blijven dus even belangrijk.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Wat lever je in?** Bewaar voor alle tien hokken de grafieken van groei en voederopname (in kg per dier per dag) en voederconversie (in kg voeder per kg groei). De afsluitcel schrijft daarnaast `hok_voederstrategie.csv` (met alleen hoknummer en gekozen strategie), `hokhypothesen.csv` (jouw argumenten en zekerheid) en `bedrijfsadvies.txt` (jouw gekozen strategie en toelichting). Samen met dit uitgevoerde notebook en de automatische controlebestanden vormen ze je volledige inlevering: een apart verslag hoeft niet.

</div>

In [ ]:
toegelaten = {"HRE10", "HRE18", "LRE10", "LRE18"}
assert gekozen_strategie in toegelaten and len(bedrijfsadvies.strip()) >= 80
assert set(hypothesen.hok) == set(perioden.hok) and not hypothesen.hok.duplicated().any()
assert hypothesen.strategie.isin(toegelaten).all()
assert hypothesen.zekerheid.isin({"laag", "matig", "hoog"}).all()
assert hypothesen.argument.str.len().ge(30).all(), "Geef voor ieder hok een inhoudelijk argument."
assert hypothesen.strategie.value_counts().to_dict() == {"HRE18": 3, "LRE18": 3, "HRE10": 2, "LRE10": 2}
uitvoermap = inlever / "resultaten_08"
uitvoermap.mkdir(exist_ok=True)
for naam in ("hokintervallen", "dierintervallen", "hokperioden", "dierperioden", "audit"):
    resultaat[naam].to_csv(uitvoermap / (naam + ".csv"), index=False, float_format="%.12g", lineterminator="\n")
hypothesen.to_csv(uitvoermap / "hokhypothesen.csv", index=False, lineterminator="\n")
hypothesen[["hok", "strategie"]].rename(columns={"hok": "hoknummer", "strategie": "voederstrategie"}).to_csv(uitvoermap / "hok_voederstrategie.csv", index=False)
fasen[fasekolommen].to_csv(uitvoermap / "voederfasen.csv", index=False)
(uitvoermap / "bedrijfsadvies.txt").write_text(gekozen_strategie + "\n\n" + bedrijfsadvies + "\n", encoding="utf-8")
clustergegevens.to_csv(uitvoermap / "groei_tegen_conversie.csv", index=False)
uitsluitingen.to_csv(uitvoermap / "uitsluitingen_voederfasen.csv", index=False)
snapshotinfo = snapshot(analysepad, inlever / "08_analysedatabase.db.gz")
assert snapshotinfo["bytes"] < 95_000_000, "Snapshot te groot voor gewone Git-opslag. Vraag hulp."
lees_basis(inlever / "beginsituatie_07")
verslag = {
    "deelnemerscode": deelnemer, "branch": branch, "rol": rol,
    "gecontroleerd_op_utc": datetime.now(timezone.utc).isoformat(),
    "bronnen": eindmanifest, "excel_import": importcontrole, "excel_herhaling": herhaalimport,
    "mqtt_bevestigd": gepubliceerd, "server_gelijk_aan_bronnen": servermeta["bronnen"] == verwacht,
    "api_import": api_import, "api_herhaling": api_herhaling,
    "basis_ongewijzigd": True, "methode": basisinfo["methode"],
    "bruikbare_dieren_per_interval": resultaat["dierintervallen"].groupby("interval").bruikbaar.sum().astype(int).to_dict(),
    "gekozen_hokken": mijn_hokken, "gekozen_periode": mijn_periode,
    "gekozen_strategie": gekozen_strategie, "bedrijfsadvies": bedrijfsadvies,
    "snapshot": snapshotinfo,
    "resultaatbestanden": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(uitvoermap.iterdir()) if p.is_file()},
}
(inlever / "08_eindcontrole.json").write_text(json.dumps(verslag, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Bewaard: tabellen, drie figuren, hokhypothesen, advies, databasesnapshot en 08_eindcontrole.json.")
print("Snapshot:", round(snapshotinfo["bytes"] / 1_000_000, 1), "MB")

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Source Control)

**Stap 1.** Kies **File → Save All** en controleer of al je antwoorden, je eigen hokkeuze, de tabel met hokhypothesen en het bedrijfsadvies ingevuld zijn, en of alle tien hokken op de eindgrafieken staan.

**Stap 2.** Open **Source Control** en selecteer dit notebook plus alle nieuwe inleverbestanden in `gegevens/inlevering/` (behoud ook `beginsituatie_07/`). Werkbestanden uit `gegevens/werk/` en wachtwoorden horen niet in Git. Gebruik bijvoorbeeld als commitbericht `Notebook 08: aanvulling en voederadvies`.

**Stap 3.** Kies **Commit** en daarna **Push** (of **Sync Changes**). Controleer vervolgens op GitHub of jouw branch en alle nieuwe bestanden echt zijn aangekomen: alleen opslaan in Codespaces is niet genoeg.

**Stap 4.** Laat weten dat je de hele reeks hebt afgerond, en zorg dat je snapshot en resultaten op GitHub staan voordat de oefenservers later worden opgeruimd.

</div>

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;white-space:normal;"><strong>Stop je Codespace</strong><br>Stop na het opslaan en pushen je Codespace via GitHub → Codespaces → de drie puntjes → <strong>Stop codespace</strong>, zodat hij geen uren blijft verbruiken. Een DigitalOcean-Droplet stopt niet vanzelf en kan ook uitgeschakeld kosten blijven opleveren: verwijder die volgens de afspraak met de docent.</div>

## Problemen oplossen

| Melding of waarneming | Wat controleer je? |
|---|---|
| Excel-bestand gewijzigd | Download de originele versie opnieuw uit de repository. Sla lange `ROW_ID`-codes niet als getallen op in Excel. |
| Bestaande meting verschilt | Behoud je werkbestand en back-up en vraag hulp voordat je iets overschrijft. |
| Serverinhoud verschilt | Controleer of beide ontvangers draaien, beide publicaties afgerond zijn en het juiste controlebestand (`08_partner_broncontrole.json`) is geüpload. |
| API niet bereikbaar of HTTP 401 | Controleer de diensten, `ca.crt` en je **`API-Droplet wachtwoord`** volgens notebook 06. |
| Beginsituatie verschilt | Overschrijf `beginsituatie_07/` niet, maar controleer of je bronrecords of rekenregels veranderd zijn. |
| Een conversie ontbreekt | Kijk in `dierintervallen` naar de kolom `reden`. Geen registratie is niet hetzelfde als nul opname. |
| Geen duidelijke hokgroepen | Beschrijf eerlijk wat je wel ziet en waar twijfel zit. De grafieken geven een onderbouwde hypothese, geen zeker bewijs. |